In [1]:
import json
from collections import defaultdict

baseline_reward_score_csv_path = "/data_center/data2/dataset/chenwy/21164-data/diffusion-dpo/sd-3-5-medium/generate_images/pick_a_pic_v2/DiffusionNFT/ckpt-0/evaluation_results.jsonl"
method_reward_score_csv_path = "/data_center/data2/dataset/chenwy/21164-data/diffusion-dpo/sd-3-5-medium/generate_images/pick_a_pic_v2/DiffusionNFT-next-top_512_images_no_anime_colorfulness_pickscore_002-hpdv3_all-inpainting/ckpt-300/evaluation_results.jsonl"

baseline_data = {}
with open(baseline_reward_score_csv_path, 'r') as f:
    for line in f:
        data = json.loads(line.strip())
        baseline_data[data['sample_id']] = data

method_data = {}
with open(method_reward_score_csv_path, 'r') as f:
    for line in f:
        data = json.loads(line.strip())
        method_data[data['sample_id']] = data

print(f"baseline: {baseline_reward_score_csv_path}")
print(f"method: {method_reward_score_csv_path}")
print(f"Baseline samples: {len(baseline_data)}")
print(f"Method samples: {len(method_data)}")

baseline: /data_center/data2/dataset/chenwy/21164-data/diffusion-dpo/sd-3-5-medium/generate_images/pick_a_pic_v2/DiffusionNFT/ckpt-0/evaluation_results.jsonl
method: /data_center/data2/dataset/chenwy/21164-data/diffusion-dpo/sd-3-5-medium/generate_images/pick_a_pic_v2/DiffusionNFT-next-top_512_images_no_anime_colorfulness_pickscore_002-hpdv3_all-inpainting/ckpt-300/evaluation_results.jsonl
Baseline samples: 500
Method samples: 500


In [2]:
if baseline_data and method_data:
    common_sample_ids = set(baseline_data.keys()) & set(method_data.keys())
    print(f"Common sample IDs: {len(common_sample_ids)}")
    
    if common_sample_ids:
        first_id = list(common_sample_ids)[0]
        eval_models = list(baseline_data[first_id]['scores'].keys())
        print(f"Evaluation models: {eval_models}")
    else:
        print("Warning: No common sample IDs found!")
        eval_models = []
else:
    eval_models = []

Common sample IDs: 500
Evaluation models: ['pickscore', 'avg', 'imagereward', 'hpsv3', 'deqa', 'aesthetic_v2_5', 'dinov2', 'q-align', 'aesthetic', 'vila_score', 'ma-agiqa', 'unifiedreward']


In [3]:
win_rates = {}
win_counts = {}
tie_counts = {}
loss_counts = {}
total_counts = {}

for model_name in eval_models:
    wins = 0
    ties = 0
    losses = 0
    total = 0
    
    for sample_id in common_sample_ids:
        if sample_id in baseline_data and sample_id in method_data:
            baseline_score = baseline_data[sample_id]['scores'].get(model_name)
            method_score = method_data[sample_id]['scores'].get(model_name)
            
            if baseline_score is not None and method_score is not None:
                total += 1
                if method_score > baseline_score:
                    wins += 1
                elif method_score == baseline_score:
                    ties += 1
                else:
                    losses += 1
    
    # Win rate 计算：wins 计 1.0，ties 计 0.5，losses 计 0.0
    win_rate = (wins + 0.5 * ties) / total if total > 0 else 0.0
    win_rates[model_name] = win_rate
    win_counts[model_name] = wins
    tie_counts[model_name] = ties
    loss_counts[model_name] = losses
    total_counts[model_name] = total
    
    print(f"{model_name}:")
    print(f"  Win rate: {win_rate:.4f} (({wins} + 0.5 * {ties}) / {total})")
    print(f"  Ties: {ties}, Losses: {losses}")
    print()


pickscore:
  Win rate: 0.4840 ((242 + 0.5 * 0) / 500)
  Ties: 0, Losses: 258

avg:
  Win rate: 0.5090 ((52 + 0.5 * 405) / 500)
  Ties: 405, Losses: 43

imagereward:
  Win rate: 0.5260 ((263 + 0.5 * 0) / 500)
  Ties: 0, Losses: 237

hpsv3:
  Win rate: 0.6700 ((335 + 0.5 * 0) / 500)
  Ties: 0, Losses: 165

deqa:
  Win rate: 0.7000 ((335 + 0.5 * 30) / 500)
  Ties: 30, Losses: 135

aesthetic_v2_5:
  Win rate: 0.3680 ((184 + 0.5 * 0) / 500)
  Ties: 0, Losses: 316

dinov2:
  Win rate: 0.3760 ((188 + 0.5 * 0) / 500)
  Ties: 0, Losses: 312

q-align:
  Win rate: 0.5980 ((231 + 0.5 * 136) / 500)
  Ties: 136, Losses: 133

aesthetic:
  Win rate: 0.5660 ((283 + 0.5 * 0) / 500)
  Ties: 0, Losses: 217

vila_score:
  Win rate: 0.4460 ((223 + 0.5 * 0) / 500)
  Ties: 0, Losses: 277

ma-agiqa:
  Win rate: 0.5680 ((284 + 0.5 * 0) / 500)
  Ties: 0, Losses: 216

unifiedreward:
  Win rate: 0.5090 ((52 + 0.5 * 405) / 500)
  Ties: 405, Losses: 43



In [ ]:
# 以表格形式展示结果
import pandas as pd

results_df = pd.DataFrame({
    'Evaluation Model': list(win_rates.keys()),
    'Win Rate': [win_rates[model] for model in win_rates.keys()],
    'Wins': [win_counts[model] for model in win_rates.keys()],
    'Ties': [tie_counts[model] for model in win_rates.keys()],
    'Losses': [loss_counts[model] for model in win_rates.keys()],
    'Total': [total_counts[model] for model in win_rates.keys()],
})

results_df = results_df.sort_values('Win Rate', ascending=False)
print(f"baseline_reward_score_csv_path: {baseline_reward_score_csv_path}")
print(f"method_reward_score_csv_path: {method_reward_score_csv_path}")
print("\nWin Rates Summary:")
print(results_df.to_string(index=False))

/data_center/data2/dataset/chenwy/21164-data/diffusion-dpo/sd-3-5-medium/generate_images/pick_a_pic_v2/DiffusionNFT-next-top_512_images_no_anime_colorfulness_pickscore_002-hpdv3_all-inpainting/ckpt-300/evaluation_results.jsonl

Win Rates Summary:
Evaluation Model  Win Rate  Wins  Ties  Losses  Total
            deqa     0.700   335    30     135    500
           hpsv3     0.670   335     0     165    500
         q-align     0.598   231   136     133    500
        ma-agiqa     0.568   284     0     216    500
       aesthetic     0.566   283     0     217    500
     imagereward     0.526   263     0     237    500
             avg     0.509    52   405      43    500
   unifiedreward     0.509    52   405      43    500
       pickscore     0.484   242     0     258    500
      vila_score     0.446   223     0     277    500
          dinov2     0.376   188     0     312    500
  aesthetic_v2_5     0.368   184     0     316    500
